# 2.3 Memory / Document Clustering — CLO04 (Unsupervised Clustering)

**Project M.O.N.D.A.Y** · Course 24CSE0316 · Labs 21–26 (K-Means, DBSCAN, Hierarchical Clustering, PCA).

**Goal.** Group stored user memories and document context into topic clusters to enable fast semantic memory retrieval and proactive suggestions. We formally compare **K-Means**, **Agglomerative Clustering**, and **DBSCAN**.

**CLO04 evidence produced here:** Silhouette scores per method and 2D post-PCA cluster visualization plots.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from ml.memory_clustering import (
    generate_memory_embeddings,
    MEMORY_TOPICS,
    N_TOPICS
)

## 1. Load Synthetic Memory Embeddings

Simulates 64-dimensional vector embeddings generated from memory items across 7 topic domains.

In [ ]:
embeddings, true_labels, sample_texts = generate_memory_embeddings(n_samples=700, seed=42)
print('Embeddings matrix shape:', embeddings.shape)
print('Number of true ground topics:', N_TOPICS)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(embeddings)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'PCA Explained Variance Ratio: {pca.explained_variance_ratio_}')

## 2. Model Training & Comparison (CLO04)

Evaluating **K-Means** (k=7), **Agglomerative Clustering** (k=7), and **DBSCAN** using Silhouette Score.

In [ ]:
models = {
    'K-Means (k=7)': KMeans(n_clusters=N_TOPICS, random_state=42, n_init=10),
    'Agglomerative (k=7)': AgglomerativeClustering(n_clusters=N_TOPICS),
    'DBSCAN (eps=1.8)': DBSCAN(eps=1.8, min_samples=5)
}

results = []
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, model) in enumerate(models.items()):
    labels = model.fit_predict(X_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    
    if len(set(labels)) > 1:
        sil_score = silhouette_score(X_scaled, labels)
    else:
        sil_score = -1.0
        
    results.append({
        'Algorithm': name,
        'Clusters Found': n_clusters,
        'Silhouette Score': sil_score
    })
    
    ax = axes[i]
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', alpha=0.7, s=20)
    ax.set_title(f'{name}\nSilhouette: {sil_score:.3f}')
    ax.set_xlabel('PCA Component 1')
    ax.set_ylabel('PCA Component 2')

results_df = pd.DataFrame(results).set_index('Algorithm')
display(results_df)
plt.tight_layout()
plt.show()

## 3. Conclusion & Production Choice

K-Means and Agglomerative Clustering perform best for cohesive topic separation with distinct Silhouette scores, powering the M.O.N.D.A.Y Memory Agent.